# Tredence AI Engineering Case Study

**Author:** Vansh Agrawal  
**Roll No:** RA2311026010120  

---

## Overview

This notebook demonstrates **learned structured pruning** applied to a fully-connected neural network trained on the **CIFAR-10** dataset.

The key idea is to attach learnable *gate scores* to every weight in each linear layer. These gates, passed through a **sigmoid** function, act as soft masks that can learn to "turn off" (prune) unimportant connections during training — without any hard threshold modifications.

We explore how the **sparsity regularization strength (λ)** affects the trade-off between **classification accuracy** and **model sparsity**.

---
## Section 1: Environment Setup

Install required packages and import all dependencies.

In [20]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd

# ── Reproducibility ────────────────────────────────────────────────────────────
torch.manual_seed(42)

# ── Device Configuration ────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


---
## Section 2: Data Loading & Preprocessing

We load **CIFAR-10** with proper normalization.

The normalization values `mean=(0.4914, 0.4822, 0.4465)` and `std=(0.2023, 0.1994, 0.2010)` are the **channel-wise statistics computed over the full CIFAR-10 training set**. Normalizing inputs to zero mean and unit variance helps the optimizer converge faster and improves generalization.

In [22]:
# ── CIFAR-10 Statistics (pre-computed over training set) ──────────────────────
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

# Build transform pipeline: convert to tensor then normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

# Download and load datasets
train_data = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_data  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True,  num_workers=0)
test_loader  = torch.utils.data.DataLoader(test_data,  batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples : {len(train_data):,}")
print(f"Test samples     : {len(test_data):,}")

Training samples : 50,000
Test samples     : 10,000


---
## Section 3: The PrunableLinear Layer

### What is `PrunableLinear`?

`PrunableLinear` is a custom `nn.Module` that extends a standard linear (fully-connected) layer with a set of **learnable gate scores** — one per weight element.

During the forward pass:
1. `gate_scores` are passed through a **sigmoid** function, squashing them to the range `(0, 1)`.
2. The computed **gates** are element-wise multiplied with the weight matrix, producing a *masked* (pruned) weight.
3. A regular linear transformation is then applied using these pruned weights.

### Why Sigmoid for Gates?

The **sigmoid** function is chosen because:
- It outputs values in `(0, 1)`, making gates natural *soft masks* (0 = off/pruned, 1 = fully active).
- It is **differentiable everywhere**, allowing gradients to flow through the gating mechanism.
- Combined with L1 regularization on the gate values, the optimizer is incentivized to push gates toward 0, achieving sparsity.

Unlike hard binary pruning (which is non-differentiable), this soft-gating approach allows the model to learn **which connections to prune** end-to-end via standard backpropagation.

In [23]:
class PrunableLinear(nn.Module):
    """
    A linear layer augmented with learnable gate scores.

    Each weight w_ij has a corresponding gate score g_ij.
    During forward, the effective weight becomes:
        w_ij_effective = sigmoid(g_ij) * w_ij

    Gate scores near -∞ → sigmoid → ~0  → weight is pruned (inactive)
    Gate scores near +∞ → sigmoid → ~1  → weight is fully active
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        # Learnable weight matrix (small random init prevents saturation)
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)

        # Learnable bias
        self.bias = nn.Parameter(torch.zeros(out_features))

        # Learnable gate scores — same shape as weight matrix
        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute soft gates in (0, 1)
        gates = torch.sigmoid(self.gate_scores)

        # Element-wise mask: pruned weights are multiplied by near-zero gates
        pruned_weights = self.weight * gates

        return F.linear(x, pruned_weights, self.bias)

---
## Section 4: Model Architecture — `PrunableNet`

A 3-layer fully-connected network where **every linear layer is replaced by a `PrunableLinear` layer**.

| Layer | Input Dim | Output Dim | Activation |
|-------|-----------|------------|------------|
| fc1   | 3×32×32 = 3072 | 512 | ReLU |
| fc2   | 512        | 256        | ReLU       |
| fc3   | 256        | 10         | — (logits) |

In [24]:
class PrunableNet(nn.Module):
    """3-layer fully-connected network with learnable gate-based pruning."""

    def __init__(self):
        super().__init__()

        self.fc1 = PrunableLinear(32 * 32 * 3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Flatten spatial dimensions: (B, C, H, W) → (B, C*H*W)
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)          # Raw logits (no softmax; CrossEntropyLoss handles it)

        return x

---
## Section 5: Sparsity Loss (L1 Regularization on Gates)

### Why L1 Regularization Leads to Sparsity

L1 regularization penalizes the **sum of absolute values** of the gate activations:

$$\mathcal{L}_{sparsity} = \lambda \sum_{i,j} \sigma(g_{ij})$$

where $\sigma$ is the sigmoid function and $\lambda$ is the regularization strength.

**Key insight:** Unlike L2 (which penalizes large values but rarely reaches exactly zero), L1 regularization creates a **constant gradient pressure** that persistently pushes small gate values toward zero. This is analogous to the "lasso" effect in classical statistics — L1 promotes *exact* zeros (sparse solutions), while L2 promotes *small but non-zero* values.

As a result, gates with low utility (i.e., connections that don't meaningfully reduce classification loss) will be driven to zero and effectively pruned.

In [25]:
def sparsity_loss(model: nn.Module) -> torch.Tensor:
    """
    Compute L1 sparsity loss: sum of all sigmoid gate activations
    across every PrunableLinear layer in the model.

    Minimizing this loss encourages gates to be pushed toward 0 (pruned).

    Args:
        model: The PrunableNet instance.

    Returns:
        Scalar tensor representing the total sparsity regularization cost.
    """
    loss = torch.tensor(0.0, device=next(model.parameters()).device)
    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)
            loss = loss + gates.sum()
    return loss

---
## Section 6: Utility Functions

Reusable helper functions for training, evaluation, and sparsity measurement.

In [26]:
def train_model(lambda_val: float, epochs: int = 10) -> nn.Module:
    """
    Initialize and train a PrunableNet with a given sparsity regularization strength.

    The total loss at each step is:
        L_total = L_classification (CrossEntropy) + lambda_val * L_sparsity (L1 on gates)

    Args:
        lambda_val: Weight for the sparsity regularization term.
        epochs: Number of training epochs.

    Returns:
        Trained PrunableNet model.
    """
    model = PrunableNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'═'*60}")
    print(f"  Training with λ = {lambda_val:.0e}")
    print(f"{'═'*60}")

    for epoch in range(epochs):
        model.train()
        total_loss     = 0.0
        total_cls_loss = 0.0
        total_sps_loss = 0.0

        for x, y in train_loader:
            # Move batch to device
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(x)

            # Compute individual loss components
            cls_loss = criterion(outputs, y)
            sps_loss = sparsity_loss(model)
            loss     = cls_loss + lambda_val * sps_loss

            # Backward & update
            loss.backward()
            optimizer.step()

            # Accumulate for logging
            total_loss     += loss.item()
            total_cls_loss += cls_loss.item()
            total_sps_loss += sps_loss.item()

        # Epoch-wise log
        print(
            f"  Epoch [{epoch+1:2d}/{epochs}] "
            f"Total Loss: {total_loss:10.4f}  "
            f"Cls Loss: {total_cls_loss:10.4f}  "
            f"Sparsity Loss: {total_sps_loss:10.4f}"
        )

    return model


def evaluate_model(model: nn.Module) -> float:
    """
    Evaluate classification accuracy on the CIFAR-10 test set.

    Args:
        model: Trained PrunableNet (or any compatible model).

    Returns:
        Accuracy as a percentage (0–100).
    """
    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, dim=1)
            total   += y.size(0)
            correct += (predicted == y).sum().item()

    return 100.0 * correct / total


def calculate_sparsity(model: nn.Module, threshold: float = 1e-2) -> float:
    """
    Compute the percentage of near-zero gates across all PrunableLinear layers.

    A gate is considered "pruned" if its value (after sigmoid) is below `threshold`.

    Args:
        model: Trained PrunableNet.
        threshold: Gate value below which a connection is considered pruned.

    Returns:
        Sparsity as a percentage (0–100).
    """
    total  = 0
    pruned = 0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates   = torch.sigmoid(module.gate_scores)
            total  += gates.numel()
            pruned += (gates < threshold).sum().item()

    return 100.0 * pruned / total

---
## Section 7: Experiments — Sweeping λ Values

We train three separate models with different sparsity regularization strengths:

| λ (lambda) | Expected Effect |
|---|---|
| `1e-5` | Very weak regularization → high accuracy, low sparsity |
| `1e-4` | Moderate regularization → balanced trade-off |
| `1e-3` | Strong regularization → high sparsity, potentially lower accuracy |

Each experiment runs for **10 epochs** with the **Adam optimizer** at `lr=1e-3`.

In [27]:
# Lambda values to sweep
lambda_values = [1e-5, 1e-4, 1e-3]

# Store results and trained models for later visualization
results       = []
trained_models = []

for lam in lambda_values:
    # Train the model
    model = train_model(lambda_val=lam, epochs=10)

    # Evaluate on test set
    accuracy = evaluate_model(model)

    # Compute sparsity
    sparsity = calculate_sparsity(model)

    results.append({
        "Lambda":   lam,
        "Accuracy (%)":   round(accuracy, 2),
        "Sparsity (%)":   round(sparsity, 4)
    })
    trained_models.append(model)

    print(f"\n  → Test Accuracy : {accuracy:.2f}%")
    print(f"  → Sparsity      : {sparsity:.4f}%")


════════════════════════════════════════════════════════════
  Training with λ = 1e-05
════════════════════════════════════════════════════════════
  Epoch [ 1/10] Total Loss:  7044.2216  Cls Loss:  1298.8827  Sparsity Loss: 574533902.5625
  Epoch [ 2/10] Total Loss:  5385.0931  Cls Loss:  1103.5778  Sparsity Loss: 428151540.0938
  Epoch [ 3/10] Total Loss:  4311.6123  Cls Loss:   999.7206  Sparsity Loss: 331189175.8125
  Epoch [ 4/10] Total Loss:  3610.3321  Cls Loss:   915.1029  Sparsity Loss: 269522932.9375
  Epoch [ 5/10] Total Loss:  3138.9899  Cls Loss:   843.6764  Sparsity Loss: 229531356.3438
  Epoch [ 6/10] Total Loss:  2800.9959  Cls Loss:   777.9455  Sparsity Loss: 202305048.7656
  Epoch [ 7/10] Total Loss:  2545.0508  Cls Loss:   716.2924  Sparsity Loss: 182875837.8125
  Epoch [ 8/10] Total Loss:  2342.7936  Cls Loss:   658.8377  Sparsity Loss: 168395596.5625
  Epoch [ 9/10] Total Loss:  2176.3427  Cls Loss:   604.4012  Sparsity Loss: 157194155.8438
  Epoch [10/10] Total L

---
## Section 8: Gate Distribution Visualization

### Interpreting the Gate Distribution Plot

The histogram below shows the distribution of **sigmoid gate values** (ranging 0 to 1) across all weights in the final trained model (λ = `1e-3`).

**What to look for:**
- **A spike near 0:** Indicates a large proportion of effectively pruned connections. These gates have been pushed toward zero by the L1 sparsity loss.
- **A mass near 1:** These connections are considered *important* — the model has chosen to keep them active.
- **A bimodal distribution** (spikes at both 0 and 1) is the ideal outcome of learned pruning — a clear separation between active and inactive connections.

With very low λ, the distribution will be more uniform (few true zeros). With high λ, more gates cluster near 0, indicating aggressive pruning.

In [28]:
fig, axes = plt.subplots(1, len(trained_models), figsize=(16, 4), sharey=True)
fig.suptitle("Gate Value Distributions Across λ Values", fontsize=14, fontweight="bold", y=1.02)

for ax, model, lam in zip(axes, trained_models, lambda_values):
    # Collect all gate values from every PrunableLinear layer
    all_gates = []
    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores).detach().cpu().numpy().flatten()
            all_gates.extend(gates)

    ax.hist(all_gates, bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
    ax.set_title(f"λ = {lam:.0e}", fontsize=12)
    ax.set_xlabel("Gate Value (sigmoid output)", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.axvline(x=0.01, color="red", linestyle="--", linewidth=1.2, label="Prune threshold (0.01)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("gate_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gate distribution plot saved to gate_distribution.png")


═══════════════════════════════════════════════════════
           EXPERIMENT RESULTS SUMMARY
═══════════════════════════════════════════════════════
Lambda  Accuracy (%)  Sparsity (%)
 1e-05         56.31        6.2485
 1e-04         56.59       33.5494
 1e-03         54.31       57.0576
═══════════════════════════════════════════════════════


,Lambda,Accuracy (%),Sparsity (%)
0,1e-05,56.31,6.2485
1,1e-04,56.59,33.5494
2,1e-03,54.31,57.0576


**Observation:**  
As λ increases, more gates cluster near 0 (red dashed line = prune threshold), confirming that stronger L1 regularization drives more connections to be inactive. This is consistent with the **lasso-like** behavior expected from L1 penalties.

---
## Section 9: Results Summary

### λ vs. Accuracy vs. Sparsity Trade-off

A fundamental tension exists in learned pruning:

- **Higher λ** → stronger push to zero gates → **more sparsity** → potentially **lower accuracy** (fewer active connections = reduced model capacity).
- **Lower λ** → weak regularization → model retains most connections → **higher accuracy** but **less sparsity** (model is not efficiently compressed).

The optimal λ lies at the *knee of the curve* — where sparsity is maximized with minimal accuracy degradation. This sweet spot is application-dependent and is typically found via hyperparameter search (as done here).

In deployment scenarios, a higher sparsity model:
- Requires **less memory**
- Enables **faster inference** (with sparse matrix operations)
- Is more suitable for **edge devices** with constrained resources

In [29]:
# Convert results to a DataFrame for clean display
df_results = pd.DataFrame(results)
df_results["Lambda"] = df_results["Lambda"].apply(lambda x: f"{x:.0e}")

print("\n" + "═" * 55)
print("           EXPERIMENT RESULTS SUMMARY")
print("═" * 55)
print(df_results.to_string(index=False))
print("═" * 55)

# Display as styled DataFrame
df_results


═══════════════════════════════════════════════════════
             FINAL MODEL PERFORMANCE
═══════════════════════════════════════════════════════
  λ = 1e-05  →  Test Accuracy:  56.31%   Sparsity:  6.2485%
  λ = 1e-04  →  Test Accuracy:  56.59%   Sparsity: 33.5494%
  λ = 1e-03  →  Test Accuracy:  54.31%   Sparsity: 57.0576%
═══════════════════════════════════════════════════════

  Best Accuracy : λ=1e-04  → 56.59%
  Most Sparse   : λ=1e-03  → 57.0576% pruned

═══════════════════════════════════════════════════════

Full Results Table:


,Lambda,Accuracy (%),Sparsity (%)
0,1e-05,56.31,6.2485
1,1e-04,56.59,33.5494
2,1e-03,54.31,57.0576


---
## Section 10: Final Output

Print the final accuracy and sparsity for each trained model in a clean format, and display the complete results DataFrame.

In [30]:
print("\n" + "═" * 55)
print("             FINAL MODEL PERFORMANCE")
print("═" * 55)

for r in results:
    print(
        f"  λ = {r['Lambda']:.0e}  →  "
        f"Test Accuracy: {r['Accuracy (%)']:6.2f}%   "
        f"Sparsity: {r['Sparsity (%)']:7.4f}%"
    )

print("═" * 55)

# Identify best accuracy and best sparsity models
best_acc     = max(results, key=lambda r: r["Accuracy (%)"])
best_sparse  = max(results, key=lambda r: r["Sparsity (%)"])

print(f"\n  Best Accuracy : λ={best_acc['Lambda']:.0e}  → {best_acc['Accuracy (%)']:.2f}%")
print(f"  Most Sparse   : λ={best_sparse['Lambda']:.0e}  → {best_sparse['Sparsity (%)']:.4f}% pruned")
print("\n" + "═" * 55)

# Final DataFrame display
print("\nFull Results Table:")
df_results


═══════════════════════════════════════════════════════
           EXPERIMENT RESULTS SUMMARY
═══════════════════════════════════════════════════════
Lambda  Accuracy (%)  Sparsity (%)
 1e-05         56.31        6.2485
 1e-04         56.59       33.5494
 1e-03         54.31       57.0576
═══════════════════════════════════════════════════════


,Lambda,Accuracy (%),Sparsity (%)
0,1e-05,56.31,6.2485
1,1e-04,56.59,33.5494
2,1e-03,54.31,57.0576


---
## Conclusion

This notebook demonstrated **learned structured pruning** via learnable gate scores on a CIFAR-10 classification task.

### Key Takeaways:

1. **`PrunableLinear`** augments standard linear layers with learnable soft gates (sigmoid-activated), enabling end-to-end differentiable pruning.
2. **Sigmoid gates** provide a smooth, bounded mask that allows gradients to flow and gates to be pushed toward 0 by the optimizer.
3. **L1 regularization** on gate activations creates constant gradient pressure that promotes exact zeros — achieving true sparsity.
4. **Higher λ** → more sparsity but risks lower accuracy; optimal λ depends on the deployment constraints.
5. The **gate distribution histogram** provides a visual diagnostic of how many connections have been effectively pruned.

This approach is a lightweight, training-time alternative to post-hoc pruning strategies and is well-suited for scenarios requiring model compression with minimal re-training overhead.

---
*Submitted for Tredence AI Engineering Case Study*  
*Vansh Agrawal | RA2311026010120*